In [1]:
import os
import cv2
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

In [2]:
DATASET_PATH = "data/train/"

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 
                                     'haarcascade_frontalface_default.xml')

X = []
y = []

for person in os.listdir(DATASET_PATH):
    person_path = os.path.join(DATASET_PATH, person)

    if not os.path.isdir(person_path):
        continue

    for img_name in os.listdir(person_path):
        img_path = os.path.join(person_path, img_name)

        img = cv2.imread(img_path)
        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        faces = face_cascade.detectMultiScale(gray, 1.3, 5)

        for (x, y_, w, h) in faces:
            face = gray[y_:y_+h, x:x+w]
            face = cv2.resize(face, (100, 100))

            X.append(face.flatten())
            y.append(person)

print(f"Dataset Loaded: {len(X)} samples")

Dataset Loaded: 75 samples


In [3]:
X = np.array(X)
y = np.array(y)

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Classes:", le.classes_)

Classes: ['ben_afflek' 'elton_john' 'jerry_seinfeld' 'madonna' 'mindy_kaling']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

In [5]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique labels:", np.unique(y))

X shape: (75, 10000)
y shape: (75,)
Unique labels: ['ben_afflek' 'elton_john' 'jerry_seinfeld' 'madonna' 'mindy_kaling']


In [6]:
model = SVC(kernel='linear', probability=True)
model.fit(X_train, y_train)

print("Training Completed")

Training Completed


In [7]:
val_path = "data/test/"

y_true = []
y_pred = []

total_images = 0
detected_faces = 0

for person in os.listdir(val_path):
    person_path = os.path.join(val_path, person)

    if not os.path.isdir(person_path):
        continue

    for img_name in os.listdir(person_path):
        img_path = os.path.join(person_path, img_name)
        total_images += 1

        img = cv2.imread(img_path)
        if img is None:
            print("Failed to load:", img_path)
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Improved detection
        faces = face_cascade.detectMultiScale(gray, 1.1, 3)

        # Fallback (VERY IMPORTANT)
        if len(faces) == 0:
            print("No face detected:", img_path)
            h, w = gray.shape
            faces = [(0, 0, w, h)]  # fallback full image

        for (x, y_, w, h) in faces:
            detected_faces += 1

            face = gray[y_:y_+h, x:x+w]
            face = cv2.resize(face, (100, 100))
            face = face.flatten().reshape(1, -1)

            pred = model.predict(face)
            label = le.inverse_transform(pred)[0]

            y_true.append(person)
            y_pred.append(label)

# Debug Summary
print("\n===== DEBUG SUMMARY =====")
print("Total images:", total_images)
print("Faces processed:", detected_faces)
print("Predictions made:", len(y_pred))

# If still empty → stop early
if len(y_pred) == 0:
    raise ValueError("No predictions made. Check face detection or dataset.")

print("\nAccuracy:", accuracy_score(y_true, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred))


===== DEBUG SUMMARY =====
Total images: 25
Faces processed: 28
Predictions made: 28

Accuracy: 0.6785714285714286

Confusion Matrix:
 [[4 0 0 0 1]
 [1 2 2 0 0]
 [0 0 6 0 0]
 [1 0 2 2 0]
 [2 0 0 0 5]]

Classification Report:
                 precision    recall  f1-score   support

    ben_afflek       0.50      0.80      0.62         5
    elton_john       1.00      0.40      0.57         5
jerry_seinfeld       0.60      1.00      0.75         6
       madonna       1.00      0.40      0.57         5
  mindy_kaling       0.83      0.71      0.77         7

      accuracy                           0.68        28
     macro avg       0.79      0.66      0.66        28
  weighted avg       0.78      0.68      0.67        28

